In [46]:
import os
import pickle
import numpy as np
np.random.seed(42)
import pandas as pd
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

In [47]:
dataset = "LastFM"
group = "tail"
sim_metric = "cos"
topk = 100

### Get the topk similar user

In [48]:
# load the llm user embedding
user_emb = pickle.load(open(os.path.join(dataset+"/handled/"+group+"_usr_emb_np.pkl"), "rb"))
print(f"{group} user embedding shape:", user_emb.shape)

tail user embedding shape: (1112, 1536)


In [49]:
# calculate the similarity score between users based on llm user embedding (streaming top-k in blocks)
from sklearn.cluster import MiniBatchKMeans
X = user_emb.astype(np.float32, copy=False)
norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
X = X / norms
N = X.shape[0]
# use fewer, more stable clusters for large N
num_bins = max(64, min(128, N // 2000))
km = MiniBatchKMeans(n_clusters=num_bins, random_state=42, batch_size=4096, n_init=3)
labels = km.fit_predict(X)
final_rank_matrix = np.empty((N, topk), dtype=np.int32)

def _topk_streaming(Xi, topk, block=2048):
    n = Xi.shape[0]
    res = np.empty((n, topk), dtype=np.int32)
    bsz = max(1, min(block, n))
    for i in range(0, n, bsz):
        j = min(i + bsz, n)
        rows = Xi[i:j]  # (b, d)
        sim = rows @ Xi.T  # (b, n)
        # exclude self
        for r in range(j - i):
            sim[r, i + r] = -np.inf
        k_eff = min(topk, n - 1)
        cand = np.argpartition(-sim, kth=k_eff, axis=1)[:, :k_eff]
        scores = np.take_along_axis(sim, cand, axis=1)
        order = np.argsort(-scores, axis=1)
        top_idx = np.take_along_axis(cand, order, axis=1)[:, :topk]
        res[i:j] = top_idx
    return res

for c in range(num_bins):
    idx = np.where(labels == c)[0]
    if idx.size == 0:
        continue
    Xi = X[idx]
    block = 4096 if idx.size >= 4096 else idx.size
    local = _topk_streaming(Xi, topk, block=block)
    final_rank_matrix[idx] = idx[local]

In [50]:
print("rank matrix shape:", final_rank_matrix.shape)

rank matrix shape: (1112, 100)


In [51]:
final_rank_matrix = final_rank_matrix

In [52]:
print("final rank matrix shape:", final_rank_matrix.shape)

final rank matrix shape: (1112, 100)


### Get the sequence length of each user

In [53]:
User = defaultdict(list)
seq_len = []
usernum, itemnum = 0, 0
f = open('./%s/handled/%s.txt' % (dataset, "inter"), 'r')
for line in f:  # use a dict to save all seqeuces of each user
    u, i = line.rstrip().split(' ')
    u = int(u)
    i = int(i)
    usernum = max(u, usernum)
    itemnum = max(i, itemnum)
    User[u].append(i)

for user, seq in User.items():
    seq_len.append(len(seq))

In [54]:
sim_user_len = []
for sim_user_list in final_rank_matrix:
    avg_len = 0
    for sim_user in sim_user_list:
        avg_len += seq_len[sim_user] / topk
    sim_user_len.append(avg_len)

In [55]:
np.mean(sim_user_len), np.mean(seq_len)

(44.16628597122304, 44.46546762589928)

### Select the similar user

In [56]:
sim_users = []
for sim_user_list in final_rank_matrix:
    sim_users.append(np.random.choice(sim_user_list, 1)[0])

In [57]:
final_rank_matrix.shape

(1112, 100)

In [58]:
final_rank_matrix[0]

array([764, 803, 818, 817, 765, 696, 763, 762, 761, 760, 759, 758, 757,
       756, 755, 754, 753, 752, 751, 750, 749, 748, 804, 747, 805, 807,
       822, 821, 826, 799, 828, 829, 830, 831, 832, 825, 827, 801, 816,
       815, 814, 813, 812, 811, 810, 809, 808, 806, 746, 745, 744, 717,
       716, 715, 714, 713, 712, 711, 710, 709, 708, 707, 706, 705, 704,
       703, 702, 701, 700, 699, 698, 766, 718, 719, 720, 721, 743, 742,
       741, 740, 739, 738, 737, 736, 735, 734, 820, 733, 731, 730, 729,
       728, 727, 726, 725, 724, 723, 722, 732, 819], dtype=int32)

In [59]:
## Save llm embedding based similar users
pickle.dump(final_rank_matrix, open(os.path.join(dataset+"/handled/"+group+"_sim_user_"+str(topk)+".pkl"), "wb"))